In [42]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  
# os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"  
from torch import nn


import numpy as np                                                                  
import torch                                          

                              
from transformers import AutoModelForCausalLM, AutoTokenizer  
from tqdm import tqdm
import matplotlib.pyplot as plt  
import torch.nn.functional as F
import gc
import re
import copy

import sys
sys.path.append('..')
import JCBScope_utils

# Move to GPU with optimal dtype
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = "cpu"


### Import pre-computed ranking

In [104]:
# mode = 'Temperature'
# mode = 'Semantic'
# mode = 'gradient_x_input'
# mode = 'Random'
mode = 'IG'
presence_list = [0.2, 0.4, 0.6, 0.8, 1.0] 

In [105]:
import json

with open("../results/Llama-3.2-1B__LOO_KL_lambada_loo_results.json") as f:
    loo_results = json.load(f)

len(loo_results['results'])

100

In [106]:
# Load the tokenizer and model

model_name = "meta-llama/Llama-3.2-1B"
model_name_short = model_name.split("/")[-1]
if device == "cpu":
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model = model.to(device)
else:
    tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="auto")
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
    
embedding_layer = model.get_input_embeddings()
embed_device = embedding_layer.weight.device    

In [107]:
front_pad = 0
back_pad = 0

front_strip = 0

# Get special tokens if available
bos_token_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.cls_token_id
eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.sep_token_id


In [108]:
if mode == 'Semantic':
    unnormalized_logits = True
else:
    unnormalized_logits = False
    

def get_influence_ranking(string):
    """Compute influence scores (Temperature or Semantic scope), return most influential token index."""
    input_ids_list = tokenizer(string, add_special_tokens=False)["input_ids"]
    if eos_token_id is not None:
        input_ids_list += [eos_token_id] * back_pad

    decoded_tokens = [tokenizer.decode([tid], skip_special_tokens=True) for tid in input_ids_list]
    grad_idx = [idx for idx in range(front_pad, len(decoded_tokens), 1)][front_strip:]
    tick_label_text = [decoded_tokens[idx] for idx in grad_idx]

    if mode == 'Random':
        most_influential_local_idx = int(np.random.randint(0, len(grad_idx)))
        most_influential_idx = grad_idx[most_influential_local_idx]
        grad_vals = np.random.random(len(grad_idx)).astype(np.float32)
        ablated_indices = np.array([most_influential_local_idx])
        return most_influential_idx, grad_vals, ablated_indices, tick_label_text, grad_idx

    input_ids = torch.tensor([input_ids_list], dtype=torch.long).to(embed_device)
    attention_mask = torch.ones_like(input_ids, device=embed_device)
    seq_len = input_ids.size(1)

    d_model = embedding_layer.embedding_dim
    residual = nn.Parameter(torch.zeros(len(grad_idx), d_model, device=embed_device))
    presence = torch.ones(len(decoded_tokens), 1, device=embed_device)
    model.eval()
    forward_pass = JCBScope_utils.customize_forward_pass(
        model, residual, presence, input_ids, grad_idx, attention_mask
    )
    hidden_norm_as_loss = (mode == 'Temperature')
    loss_position = seq_len - 2

    if mode == 'IG':
        grad_list = []
        for alpha in presence_list:
            loss, logits_orig = forward_pass(
                loss_position=loss_position,
                hidden_norm_as_loss=hidden_norm_as_loss,
                unnormalized_logits=True,
                tie_input_output_embed=False,
                alpha=alpha,
            )
            g = torch.autograd.grad(loss, residual, retain_graph=False)[0]
            grad_list.append(g.detach().clone())
            del loss
        grads = torch.stack(grad_list).mean(dim=0)
        del grad_list
        with torch.no_grad():
            token_embeds = embedding_layer(input_ids[0, grad_idx])
        grad_vals = (grads * token_embeds).norm(dim=-1).squeeze().cpu().numpy()
    else:
        loss, logits_orig = forward_pass(
            loss_position=loss_position,
            hidden_norm_as_loss=hidden_norm_as_loss,
            unnormalized_logits=True,
            tie_input_output_embed=False,
        )
        grads = torch.autograd.grad(loss, residual, retain_graph=False)[0]
        del loss
        if mode == 'gradient_x_input':
            with torch.no_grad():
                token_embeds = embedding_layer(input_ids[0, grad_idx])
            grad_vals = (grads * token_embeds).norm(dim=-1).squeeze().cpu().numpy()
        else:
            grad_vals = grads.norm(dim=-1).squeeze().cpu().numpy()
    if grad_vals.ndim > 1:
        grad_vals = grad_vals.squeeze()

    most_influential_local_idx = int(np.argmax(grad_vals))
    most_influential_idx = grad_idx[most_influential_local_idx]
    ablated_indices = np.array([most_influential_local_idx])

    # del loss, grads, logits_orig, forward_pass
    gc.collect()
    if device != "cpu":
        torch.cuda.empty_cache()

    return most_influential_idx, grad_vals, ablated_indices, tick_label_text, grad_idx

In [ ]:
import json
from pathlib import Path

# 1. Load each prompt from loo_results['results']
# 2. Use temperature/semantic scope to get most influential token index
# 3. Locate this index in ranked_token_indices (LOO ranking)
# 4. Save rank and ranking_pct into each result, with mode name in key

loo_json_path = Path("../results/Llama-3.2-1B__LOO_KL_lambada_loo_results.json")
with open(loo_json_path, "r", encoding="utf-8") as f:
    loo_results = json.load(f)

for i, item in enumerate(tqdm(loo_results["results"], desc="Processing prompts")):
    prompt = item["prompt"]
    ranked_token_indices = item["ranked_token_indices"]

    most_influential_idx, grad_vals, ablated_indices, tick_label_text, grad_idx = get_influence_ranking(prompt)

    if most_influential_idx in ranked_token_indices:
        rank = ranked_token_indices.index(most_influential_idx)
        ranking_pct = (rank / len(ranked_token_indices)) * 100
    else:
        rank = None
        ranking_pct = None

    print(prompt)
    item[f"{mode}_rank"] = rank
    item[f"{mode}_ranking_pct"] = ranking_pct
    print(f"[{i}] {mode}_rank = {rank}, {mode}_ranking_pct = {ranking_pct}")
    most_influential_token_scope = tick_label_text[grad_idx[most_influential_idx]]
    print(f"Most influential token index (by {mode} scope): {most_influential_token_scope}")
    most_influential_token_loo = tick_label_text[ranked_token_indices[0]]
    print(f"Most influential token index (by LOO): {most_influential_token_loo}")


    if (i + 1) <= 3:
        _last_grad_vals, _last_ablated, _last_tick = grad_vals, ablated_indices, tick_label_text

# Save ranking and ranking_pct back to the same JSON file
with open(loo_json_path, "w", encoding="utf-8") as f:
    json.dump(loo_results, f, indent=2, ensure_ascii=False)
print(f"Saved {len(loo_results['results'])} results with {mode}_rank and {mode}_ranking_pct to {loo_json_path}")




Processing prompts:   1%|          | 1/100 [00:05<08:55,  5.41s/it]

She had never been inside his house before. It was small and surprisingly neat for a man who lived alone. The furniture was sparse but of good quality. A number of photographs sat on the mantel. She moved closer for a better look. The first depicted a young couple holding a little boy. There were three other photos of the same couple
[0] IG_rank = 3, IG_ranking_pct = 4.3478260869565215
Most influential token index (by IG scope):  couple
Most influential token index (by LOO):  same


Processing prompts:   2%|▏         | 2/100 [00:11<09:21,  5.73s/it]

With a square of late-afternoon sun on the floor, even the red room showed itself to be what Beau had described, a dusty collection of old things. Sam took up a broom and swept the white stones and bundled herbs into a harmless pile. The stiff snake went into a garbage bag. It was a little creepy, picking it up, but she handled it just fine. She dropped the black candles—so dusty that they were nearly gray, in the clear light of day—into the same bag with the snake
[1] IG_rank = 51, IG_ranking_pct = 48.57142857142857
Most influential token index (by IG scope): ,
Most influential token index (by LOO): With


Processing prompts:   3%|▎         | 3/100 [00:17<09:13,  5.71s/it]

...

Vampire or not, I was nearly mortal during the day, and my hands felt like lead, especially after going through a few rounds on the heavy bag.

But even though sunset was still under two hours away, I had more than enough strength to hit the bag hard enough to rock the little trainer. He grunted through the shockwaves, screaming at me to keep my hands up even as he struggled to hold onto the bag
[61] IG_rank = 6, IG_ranking_pct = 6.976744186046512
Most influential token index (by IG scope):  bag
Most influential token index (by LOO):  the


Processing prompts:  63%|██████▎   | 63/100 [08:09<06:40, 10.81s/it]

The pliant branches fell back into place behind them, hiding them from the outside world like a glowing green curtain.

Jake circled behind her, as though allowing her a moment to marvel at the beauty of where he had brought her. Suddenly, he jerked her arm back, putting her off-balance. At the same time he knocked his knee into the back of hers
[62] IG_rank = 2, IG_ranking_pct = 2.666666666666667
Most influential token index (by IG scope):  knee
Most influential token index (by LOO):  of


Processing prompts:  64%|██████▍   | 64/100 [08:18<06:17, 10.48s/it]

Whatever was on the other side of the grate pushed full force, and I almost lost hold with my one hand, still holding the hatpin. Then it was cut too.
I switched hands, trying to keep the vent from coming free and prevent another laceration.
What was back there?  I pulled my feet up and pressed them against the grate
[63] IG_rank = 2, IG_ranking_pct = 2.8169014084507045
Most influential token index (by IG scope):  grate
Most influential token index (by LOO):  the


In [ ]:
## Summary: {mode} scope most influential token rank in LOO ranking
results = loo_results["results"]
rank_key, pct_key = f"{mode}_rank", f"{mode}_ranking_pct"
for i, r in enumerate(results):
    rank = r.get(rank_key)
    pct = r.get(pct_key)
    rank_str = str(rank) if rank is not None else "N/A"
    pct_str = f"{pct:.2f}%" if pct is not None else "N/A"
    print(f"[{i}] rank={rank_str}  ranking_pct={pct_str}")
    

[0] rank=0  ranking_pct=0.00%
[1] rank=2  ranking_pct=1.90%
[2] rank=1  ranking_pct=1.22%
[3] rank=8  ranking_pct=10.13%
[4] rank=23  ranking_pct=27.38%
[5] rank=0  ranking_pct=0.00%
[6] rank=1  ranking_pct=1.22%
[7] rank=3  ranking_pct=3.75%
[8] rank=1  ranking_pct=1.52%
[9] rank=1  ranking_pct=1.37%
[10] rank=3  ranking_pct=3.26%
[11] rank=1  ranking_pct=1.18%
[12] rank=1  ranking_pct=1.32%
[13] rank=2  ranking_pct=2.11%
[14] rank=5  ranking_pct=7.04%
[15] rank=7  ranking_pct=9.59%
[16] rank=58  ranking_pct=72.50%
[17] rank=2  ranking_pct=1.61%
[18] rank=1  ranking_pct=0.96%
[19] rank=1  ranking_pct=1.18%
[20] rank=2  ranking_pct=2.25%
[21] rank=5  ranking_pct=6.41%
[22] rank=3  ranking_pct=3.90%
[23] rank=2  ranking_pct=2.50%
[24] rank=3  ranking_pct=3.80%
[25] rank=8  ranking_pct=7.77%
[26] rank=2  ranking_pct=2.78%
[27] rank=4  ranking_pct=5.71%
[28] rank=1  ranking_pct=1.30%
[29] rank=19  ranking_pct=25.33%
[30] rank=2  ranking_pct=2.86%
[31] rank=1  ranking_pct=1.52%
[32] rank=3

In [82]:

from math import sqrt

# Report ranking stats: where does the most influential token (by {mode} scope) rank in LOO?
rank_key, pct_key = f"{mode}_rank", f"{mode}_ranking_pct"
ranks = [r[rank_key] for r in results if r.get(rank_key) is not None]
pcts = [r[pct_key] for r in results if r.get(pct_key) is not None]
total = len(results)

if ranks:
    avg_rank = sum(ranks) / len(ranks)
    avg_pct = sum(pcts) / len(pcts) if pcts else float("nan") 

    # Compute SEM (standard error of the mean) for mean_ranking_pct if possible
    sem_ranking_pct = None
    if pcts and len(pcts) > 1:
        mean_pct = sum(pcts) / len(pcts)
        variance_pct = sum((x - mean_pct) ** 2 for x in pcts) / (len(pcts) - 1)
        sem_ranking_pct = sqrt(variance_pct / len(pcts))

    print(f"\n{mode} scope vs LOO ranking ({len(ranks)}/{total} prompts):")
    print(f"  Mean rank of most influential token in LOO: {avg_rank:.2f}")
    if sem_ranking_pct is not None:
        print(f"  Mean ranking percentile: {avg_pct:.2f} ± {sem_ranking_pct:.2f}%")
    else:
        print(f"  Mean ranking percentile: {avg_pct:.2f}%")
else:
    print("No rank data. Run the processing cell first.")

# Calculate how often most influential token's ranking percentile is within 5% of the top
within_5_pct_count = sum(1 for r in results if r.get(pct_key) is not None and r[pct_key] <= 5)
fraction_within_5_pct = within_5_pct_count / total if total else 0
print(f"\n{mode} scope most influential token is within top 5% of LOO ranking for {within_5_pct_count}/{total} prompts ({fraction_within_5_pct:.1%})")

# Save to master_results.json
label = f"Llama-3.2-1B__{mode}_lambada_loo_rank"
master_path = Path("../results/master_results.json")
master_path.parent.mkdir(parents=True, exist_ok=True)
master = {}
if master_path.exists():
    with open(master_path, "r", encoding="utf-8") as f:
        master = json.load(f)

entry = {
    "mean_rank": sum(ranks) / len(ranks) if ranks else None,
    "mean_ranking_pct": sum(pcts) / len(pcts) if pcts else None,
    "sem_mean_ranking_pct": sem_ranking_pct,
    "n_with_rank": len(ranks),
    "total": total,
    "within_top5_pct_count": within_5_pct_count,
    "fraction_within_top5_pct": fraction_within_5_pct,
}
master[label] = entry
with open(master_path, "w", encoding="utf-8") as f:
    json.dump(master, f, indent=2)
print(f"\nSaved to {master_path} (label={label})")




gradient_x_input scope vs LOO ranking (100/100 prompts):
  Mean rank of most influential token in LOO: 5.48
  Mean ranking percentile: 6.75 ± 1.17%

gradient_x_input scope most influential token is within top 5% of LOO ranking for 69/100 prompts (69.0%)

Saved to ../results/master_results.json (label=Llama-3.2-1B__gradient_x_input_lambada_loo_rank)
